In [69]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector
from dotenv import load_dotenv
from sqlalchemy import create_engine


In [41]:
load_dotenv(override=True)

True

In [42]:
host = os.getenv("DB_HOST")
username = os.getenv("DB_USER")
password = os.getenv("DB_PASSWORD")

database = "brazil_ecommerce_db"


In [43]:
db = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database="brazil_ecommerce_db"
)
print("Connected successfully")

Connected successfully


In [44]:
cursor = db.cursor()

# Get all table names
cursor.execute("SHOW TABLES")

tables = [table[0] for table in cursor.fetchall()]

# Create df_<table_name> dynamically
for table in tables:
    df_name = f"df_{table}"

    globals()[df_name] = pd.read_sql(
        f"SELECT * FROM `{table}`",
        db
    )

    print(f"{df_name} loaded successfully")

cursor.close()
db.close()

C:\Users\Acer\AppData\Local\Temp\ipykernel_8732\3510591691.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  globals()[df_name] = pd.read_sql(


df_customers loaded successfully
df_geolocation loaded successfully
df_order_items loaded successfully
df_order_payments loaded successfully
df_order_reviews loaded successfully
df_orders loaded successfully
df_product_category_name_translation loaded successfully
df_products loaded successfully
df_sellers loaded successfully


In [45]:
df_customers = pd.DataFrame(df_customers)
df_products = pd.DataFrame(df_products)
df_geolocation = pd.DataFrame(df_geolocation)
df_order_items = pd.DataFrame(df_order_items)
df_order_payments = pd.DataFrame(df_order_payments)
df_order_reviews = pd.DataFrame(df_order_reviews)
df_orders = pd.DataFrame(df_orders)
df_product_category_name_translation = pd.DataFrame(df_product_category_name_translation)
df_sellers = pd.DataFrame(df_sellers)

In [46]:
print("="*50)
print("Customers Data")
print(df_customers.isnull().sum())
print("="*50)
print("Geolocation Data")
print(df_geolocation.isnull().sum())
print("="*50)
print("Order Item Data")
print(df_order_items.isnull().sum())
print("="*50)
print("Order PAyment Data")
print(df_order_payments.isnull().sum())
print("="*50)
print("Order Reviews Data")
print(df_order_reviews.isnull().sum())
print("="*50)
print("Order  Data")
print(df_orders.isnull().sum())
print("="*50)
print("product_category_name_translation Data")
print(df_product_category_name_translation.isnull().sum())
print("="*50)
print("products Data")
print(df_products.isnull().sum())
print("="*50)
print("Sellers Data")
print(df_sellers.isnull().sum())
print("="*50)


Customers Data
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Geolocation Data
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Order Item Data
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Order PAyment Data
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Order Reviews Data
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int6

In [47]:
df_products["product_category_name"] = (
    df_products["product_category_name"].fillna("UnKnown_Product")
)

col_to_fill_with_median = ["product_name_lenght", "product_description_lenght",
                           "product_weight_g","product_length_cm",
                           "product_height_cm", "product_width_cm"
]

for col in col_to_fill_with_median:
    df_products[col] = df_products[col].fillna(df_products[col].median())
    
    
df_products["product_photos_qty"] = df_products["product_photos_qty"].fillna(0)

In [49]:
df_orders["order_status"].unique()

<StringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

In [50]:
df_orders.groupby("order_status").size()

order_status
approved           2
canceled         625
created            5
delivered      96478
invoiced         314
processing       301
shipped         1107
unavailable      609
dtype: int64

### Since the date which are mostly based on the Status, the NaN value NaT(Not a Time)

In [51]:
cols = ["order_approved_at", "order_delivered_carrier_date", "order_delivered_customer_date"]

for col in cols:
    df_orders[col] = pd.to_datetime(df_orders[col], errors="coerce")

In [58]:
df_orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp                    str
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date               str
dtype: object

In [52]:
cols = ["review_comment_title", "review_comment_message"]

for col in cols:
    df_order_reviews[col] = df_order_reviews[col].fillna("UnAvailable")

In [61]:
print("="*50)
print("Customers Data")
print(df_customers.isnull().sum())
print("Duplication SUM")
print(df_customers.duplicated().sum())
print("="*50)
print("Geolocation Data")
print(df_geolocation.isnull().sum())
print("Duplication SUM")
print(df_geolocation.duplicated().sum())
print("="*50)
print("Order Item Data")
print(df_order_items.isnull().sum())
print("Duplication SUM")
print(df_order_items.duplicated().sum())
print("="*50)
print("Order PAyment Data")
print(df_order_payments.isnull().sum())
print("Duplication SUM")
print(df_order_payments.duplicated().sum())
print("="*50)
print("Order Reviews Data")
print(df_order_reviews.isnull().sum())
print("Duplication SUM")
print(df_order_reviews.duplicated().sum())
print("="*50)
print("Order  Data")
print(df_orders.isnull().sum())
print("Duplication SUM")
print(df_orders.duplicated().sum())
print("="*50)
print("product_category_name_translation Data")
print(df_product_category_name_translation.isnull().sum())
print("Duplication SUM")
print(df_product_category_name_translation.duplicated().sum())
print("="*50)
print("products Data")
print(df_products.isnull().sum())
print("Duplication SUM")
print(df_products.duplicated().sum())
print("="*50)
print("Sellers Data")
print(df_sellers.isnull().sum())
print("Duplication SUM")
print(df_sellers.duplicated().sum())
print("="*50)


Customers Data
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Duplication SUM
0
Geolocation Data
geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64
Duplication SUM
261831
Order Item Data
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
Duplication SUM
0
Order PAyment Data
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
Duplication SUM
0
Order Reviews Data
review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date

The Missing values are there as NaT (There is no time to put there.)

In [66]:
df_geolocation = df_geolocation.drop_duplicates()


In [78]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

url = URL.create(
    drivername="mysql+pymysql",
    username=username,
    password=password,
    host=host,
    database=database
)

engine = create_engine(url)

In [79]:
df_customers.to_sql(
    "customers",
    engine,
    if_exists="append",
    index=False
)

99441

In [80]:
df_geolocation.to_sql(
    "geolocation",
    engine,
    if_exists="append",
    index=False
)

738332

In [81]:
df_products.to_sql(
    "products",
    engine,
    if_exists="append",
    index=False
)

32951

In [82]:
df_sellers.to_sql(
    "sellers",
    engine,
    if_exists="append",
    index=False
)

3095

In [83]:
df_orders.to_sql(
    "orders",
    engine,
    if_exists="append",
    index=False
)

99441

In [84]:
df_order_items.to_sql(
    "order_items",
    engine,
    if_exists="append",
    index=False
)

112650

In [85]:
df_order_payments.to_sql(
    "order_payments",
    engine,
    if_exists="append",
    index=False
)

103886

In [86]:
df_order_reviews.to_sql(
    "order_reviews",
    engine,
    if_exists="append",
    index=False
)

99224

# **DONE!!**